<a href="https://colab.research.google.com/github/cd-uf/review/blob/main/Bertopic_Scibert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 0) Install dependencies (Colab)
# ============================================================
# If you already installed these in the current runtime, you can re-run safely.
!pip -q install bertopic umap-learn hdbscan transformers accelerate sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 2.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 1) Imports
# ============================================================
import io
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModel

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
# ============================================================
# 2) Load data (from Colab upload)
# ============================================================
# Upload your file in Colab:
#   from google.colab import files
#   uploaded = files.upload()
#
# Then choose the exact name shown in uploaded.keys().

from google.colab import files
uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

# ---- set your file name here ----
file_name = list(uploaded.keys())[0]  # uses the first uploaded file
# file_name = "667 articles (1).csv"  # or set explicitly

file_content_bytes = uploaded[file_name]
file_content_string = file_content_bytes.decode("utf-8", errors="replace")
df_articles = pd.read_csv(io.StringIO(file_content_string))

print("Loaded rows:", len(df_articles))
print("Columns:", df_articles.columns.tolist())


Saving 667 articles.csv to 667 articles.csv
Uploaded files: ['667 articles.csv']
Loaded rows: 667
Columns: ['id', 'title', 'abstract']


In [ ]:
# ============================================================
# 2b) Reproducibility controls (seeds + deterministic settings)
# ============================================================
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)


In [ ]:
# ============================================================
# 3) Prepare corpus: clean abstracts + keep traceable item IDs
# ============================================================
# We keep an item_number that you can use to trace back to the original CSV row.
# If you want 1..N numbering, we set that explicitly.
df_articles = df_articles.copy()
df_articles["item_number"] = np.arange(1, len(df_articles) + 1)

# Ensure abstract column exists
assert "abstract" in df_articles.columns, "Your CSV must have a column named 'abstract'."

# Drop missing/empty abstracts (prevents tokenizer issues and nonsense topics)
df_work = df_articles.dropna(subset=["abstract"]).copy()
df_work = df_work[df_work["abstract"].astype(str).str.strip().ne("")]

# Build docs list (aligned with df_work rows)
docs = df_work["abstract"].astype(str).tolist()

print(f"Working docs: {len(docs)} (from {len(df_articles)} total rows)")

Working docs: 667 (from 667 total rows)


In [ ]:
# ============================================================
# 4) SciBERT embedding: mean pooling over token embeddings
# ============================================================
# What this does:
# - Tokenizes each abstract
# - Runs SciBERT forward pass
# - Uses attention-mask mean pooling to get a single vector per abstract
# - Optionally L2-normalizes those vectors
#
# Performance notes:
# - If you're on CPU, this can take minutes.
# - On GPU, increase batch_size (e.g. 64) for faster throughput.
# - max_length trades speed vs capturing long abstracts (128/256 are common).


class SciBERTEmbedder:
    def __init__(
        self,
        model_name: str = "allenai/scibert_scivocab_uncased",
        device: str | None = None,
        max_length: int = 256
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.max_length = max_length

    @torch.no_grad()
    def encode(self, texts, batch_size: int = 32, normalize_embeddings: bool = True):
        all_embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

            enc = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt"
            ).to(self.device)

            out = self.model(**enc)                         # [B, T, H]
            token_embs = out.last_hidden_state
            attn_mask = enc["attention_mask"].unsqueeze(-1) # [B, T, 1]

            # Mean pooling with attention mask
            summed = (token_embs * attn_mask).sum(dim=1)    # [B, H]
            counts = attn_mask.sum(dim=1).clamp(min=1)      # [B, 1]
            sent_embs = summed / counts                     # [B, H]

            if normalize_embeddings:
                sent_embs = torch.nn.functional.normalize(sent_embs, p=2, dim=1)

            all_embs.append(sent_embs.cpu().numpy())

        return np.vstack(all_embs)


# ---- build embeddings ----
embedder = SciBERTEmbedder(max_length=256)
print("Using device:", embedder.device, "| CUDA available:", torch.cuda.is_available())

embeddings = embedder.encode(docs, batch_size=32)
print("Embeddings shape:", embeddings.shape)  # expected: (n_docs, 768)



config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Using device: cpu | CUDA available: False
Embeddings shape: (667, 768)


In [ ]:
# ============================================================
# 5) Topic representation stopwords (for titles/keywords only)
# ============================================================
# Important:
# - Stopwords here affect ONLY the c-TF-IDF representation (topic keywords/titles).
# - They do NOT affect SciBERT embeddings or clustering.
#
# Start with sklearn English stopwords, then add domain-specific terms.
custom_stopwords = {"income", "basic"}  # add more if needed: {"universal", "policy", ...}
all_stopwords = list(ENGLISH_STOP_WORDS.union(custom_stopwords))

vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    ngram_range=(1, 2)  # include unigrams + bigrams in topic representations
)



In [ ]:
# ============================================================
# 6) Granularity controls (UMAP + HDBSCAN)
# ============================================================
# These parameters largely determine how many topics you get.
#

# This preset aims for "more than a handful" of topics without being extremely fragmented.
umap_model = UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    min_samples=2,
    cluster_selection_method="eom",
    prediction_data=True
)


In [ ]:
# ============================================================
# 7) Fit BERTopic using precomputed SciBERT embeddings
# ============================================================
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)

topic_info = topic_model.get_topic_info()
n_topics_excl_outliers = (topic_info["Topic"] != -1).sum()
n_outliers = int(np.sum(np.array(topics) == -1))

print("Topics (excluding -1):", n_topics_excl_outliers)
print("Outliers (-1):", n_outliers)
print(topic_info.head(20))


2026-01-26 08:50:22,845 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-26 08:50:32,889 - BERTopic - Dimensionality - Completed ✓
2026-01-26 08:50:32,890 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-26 08:50:32,944 - BERTopic - Cluster - Completed ✓
2026-01-26 08:50:32,950 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-26 08:50:33,288 - BERTopic - Representation - Completed ✓


Topics (excluding -1): 18
Outliers (-1): 159
    Topic  Count                                         Name  \
0      -1    159              -1_social_ubi_welfare_universal   
1       0     47                   0_ubi_social_economic_time   
2       1     46                 1_social_universal_care_work   
3       2     45             2_van_work_unconditional_liberal   
4       3     42                 3_tax_model_equilibrium_rate   
5       4     41               4_ubi_support_policy_political   
6       5     38               5_social_main_universal_policy   
7       6     34            6_ubi_support_support ubi_welfare   
8       7     31              7_welfare_labour_article_policy   
9       8     31                  8_tax_poverty_families_poor   
10      9     30          9_policy_political_welfare_proposal   
11     10     22  10_pandemic_covid19_policy_covid19 pandemic   
12     11     20              11_van_islamic_published_author   
13     12     19                  12_labor_ra

In [ ]:
# ============================================================
# 8) Document-level table: topic + probability + item_number
# ============================================================
# BERTopic provides a document info table; it often includes a 'Probability' column.
doc_info = topic_model.get_document_info(docs).copy()

# Add item_number to keep traceability to the original CSV
doc_info["item_number"] = df_work["item_number"].values

# Robust probability handling:
# - probs may be None
# - probs may be shape (n_docs,) or (n_docs, n_topics)
if "Probability" not in doc_info.columns:
    if probs is None:
        doc_info["Probability"] = np.nan
    else:
        probs_arr = np.asarray(probs)
        doc_info["Probability"] = probs_arr if probs_arr.ndim == 1 else probs_arr.max(axis=1)

# Optional: attach topic to df_work / df_articles for later merges
df_work = df_work.copy()
df_work["topic_id"] = topics
df_work["topic_probability"] = doc_info["Probability"].values

df_articles = df_articles.merge(
    df_work[["item_number", "topic_id", "topic_probability"]],
    on="item_number",
    how="left"
)

print(df_articles[["item_number", "topic_id", "topic_probability"]].head())


   item_number  topic_id  topic_probability
0            1         6           0.671563
1            2         0           0.868115
2            3         7           1.000000
3            4         8           0.881857
4            5         4           1.000000


In [ ]:
# ============================================================
# 9) Export 1: Topic summary (Topic, Count, Name)
# ============================================================
topic_info.to_csv("bertopic_topic_summary.csv", index=False)



In [ ]:
# ============================================================
# 10) Export 2: Topic keywords + weights (long format)
# ============================================================
# One row per (topic, keyword), includes keyword weights.
rows = []
for topic_id in topic_info["Topic"].tolist():
    if topic_id == -1:
        continue
    for word, weight in (topic_model.get_topic(topic_id) or []):
        rows.append({
            "topic_id": int(topic_id),
            "keyword": word,
            "weight": float(weight)
        })

df_keywords = pd.DataFrame(rows)
df_keywords.to_csv("bertopic_topic_keywords.csv", index=False)


In [ ]:
# ============================================================
# 11) Export 3: Document assignments + probabilities (+ item_number)
# ============================================================
# Useful for analysis: topic prevalence by year, gender, journal, etc.
doc_info.to_csv("bertopic_documents_with_topics.csv", index=False)

In [ ]:
# ============================================================
# 12) Export 4: Combined topic export with:
#     Topic | Count | Name | Representation(with weights) | Representative_Docs(with item_number)
# ============================================================
TOP_K_WORDS = 10
TOP_K_DOCS = 5

rows = []
for _, r in topic_info.iterrows():
    topic_id = int(r["Topic"])
    count = int(r["Count"])
    name = r.get("Name", "")

    # Representation: "word (weight); word (weight); ..."
    if topic_id == -1:
        representation = ""
    else:
        kws = topic_model.get_topic(topic_id) or []
        representation = "; ".join([f"{w} ({wt:.4f})" for w, wt in kws[:TOP_K_WORDS]])

    # Representative docs: choose highest-probability docs within topic
    if topic_id == -1:
        rep_docs_str = ""
    else:
        subset = doc_info[doc_info["Topic"] == topic_id].copy()
        subset = subset.sort_values("Probability", ascending=False).head(TOP_K_DOCS)

        rep_items = []
        for _, dr in subset.iterrows():
            item_no = int(dr["item_number"])
            text = str(dr["Document"])
            snippet = " ".join(text.split())[:250]
            prob = dr.get("Probability", np.nan)
            rep_items.append(f"{item_no} (p={prob:.3f}): {snippet}")

        rep_docs_str = " | ".join(rep_items)

    rows.append({
        "Topic": topic_id,
        "Count": count,
        "Name": name,
        "Representation": representation,
        "Representative_Docs": rep_docs_str
    })

df_topic_export = pd.DataFrame(rows)
df_topic_export.to_csv("bertopic_topics_with_reps.csv", index=False)



In [ ]:
# ============================================================
# 13) Download exports from Colab
# ============================================================
# These will appear in your browser as downloads.
files.download("bertopic_topic_summary.csv")
files.download("bertopic_topic_keywords.csv")
files.download("bertopic_documents_with_topics.csv")
files.download("bertopic_topics_with_reps.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# 14) Optional: save embeddings so you never recompute them
# ============================================================
# Recomputing SciBERT embeddings is slow; caching speeds iteration.
np.save("scibert_embeddings.npy", embeddings)
files.download("scibert_embeddings.npy")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# 15) Save precomputed SciBERT embeddings for reuse
# ============================================================

from importlib.metadata import version

packages = ["bertopic", "transformers", "torch", "scikit-learn", "umap-learn", "hdbscan"]

for p in packages:
    try:
        print(p, version(p))
    except:
        print(p, "version not found")


bertopic 0.17.4
transformers 4.57.6
torch 2.9.0+cpu
scikit-learn 1.6.1
umap-learn 0.5.11
hdbscan 0.8.41


In [ ]:
##===
#Visualise

from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]